# Fine-tune Khmer XLS-R on 5,000 rows with Kaggle

This notebook builds a 5,000-row manifest while keeping the original 126-row test set unchanged. It excludes those test speakers and transcript groups from training and validation, then materializes audio, smoke-checks, trains, and evaluates the new checkpoint.

Before opening this notebook, follow kaggle/README.md to upload the source bundle as a Kaggle Dataset and attach it here. In Notebook settings, turn on GPU and Internet. Internet is required for downloading the dataset and model.

If a Kaggle session is interrupted, checkpoints can be resumed only after the model output and materialized WAV/manifests have been saved and attached to the new session. See the README for restore settings.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU accelerator in Kaggle Notebook settings before continuing.")

In [ ]:
from pathlib import Path
import shutil
import zipfile

INPUT_ROOT = Path("/kaggle/input")
BUNDLE_NAME = "khmer_asr_5k_source_bundle.zip"
PROJECT_ROOT = Path("/kaggle/working/asr_project")
RUN_ROOT = Path("/kaggle/working/asr_run")
MANIFEST_DIR = RUN_ROOT / "data" / "voice_5000_fixed_test"
BASE_MANIFEST_DIR = PROJECT_ROOT / "data" / "voice_1000"
MATERIALIZED_DIR = MANIFEST_DIR / "materialized_training"
BASE_MODEL = "vitouphy/wav2vec2-xls-r-300m-khmer"
SMOKE_DIR = RUN_ROOT / "models" / "khmer_xlsr_ddd_5000_smoke"
MODEL_DIR = RUN_ROOT / "models" / "khmer_xlsr_ddd_5000_clean"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
bundle_matches = list(INPUT_ROOT.glob(f"**/{BUNDLE_NAME}"))
if bundle_matches:
    with zipfile.ZipFile(bundle_matches[0]) as archive:
        archive.extractall(PROJECT_ROOT)
else:
    extracted_scripts = list(INPUT_ROOT.glob("**/scripts/build_asr_manifest.py"))
    if not extracted_scripts:
        raise FileNotFoundError(f"Attach {BUNDLE_NAME} or its extracted contents as a Kaggle Dataset")
    extracted_root = extracted_scripts[0].parents[1]
    shutil.copytree(extracted_root, PROJECT_ROOT, dirs_exist_ok=True)

# For a resumed run, set this to the attached prior Kaggle output folder.
# Example: Path("/kaggle/input/previous-asr-run/asr_run")
RESTORE_FROM = None
if RESTORE_FROM is not None:
    restore_root = Path(RESTORE_FROM)
    for folder_name in ("data", "models"):
        source = restore_root / folder_name
        destination = RUN_ROOT / folder_name
        if source.exists():
            shutil.copytree(source, destination, dirs_exist_ok=True)
    print("Restored prior run files from:", restore_root)

print("Project source:", PROJECT_ROOT)
print("Run output:", RUN_ROOT)

In [ ]:
import subprocess
import sys

def run_project_script(script, *arguments):
    command = [sys.executable, str(PROJECT_ROOT / script), *map(str, arguments)]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-asr.txt")],
    check=True,
)
print("ASR dependencies installed. Kaggle-provided PyTorch was left in place.")

## Build a fixed-test 5,000-row manifest

The original test CSV is copied unchanged from the 1,000-row manifest. The builder keeps those test speakers and transcript groups out of the new training and validation data. The 5,000-row target includes the fixed test rows.

In [ ]:
import json

if not (MANIFEST_DIR / "manifest_summary.json").is_file():
    run_project_script(
        "scripts/build_asr_manifest.py",
        "--target-rows", "5000",
        "--scan-rows", "25000",
        "--fixed-test-manifest-dir", BASE_MANIFEST_DIR,
        "--output-dir", MANIFEST_DIR,
    )
else:
    print("Existing fixed-test manifest found; reusing:", MANIFEST_DIR)

summary = json.loads((MANIFEST_DIR / "manifest_summary.json").read_text(encoding="utf-8"))
print("Selected:", summary["selected_rows"])
print("Split counts:", summary["split_counts"])
print("Fixed test source:", summary["fixed_test_manifest_dir"])

## Materialize the audio

The WAV files and manifests go under /kaggle/working/asr_run. Save a Kaggle notebook version after this cell finishes so the materialized data is available to attach if the runtime needs to be resumed.

In [ ]:
if not (MATERIALIZED_DIR / "manifests" / "train.jsonl").is_file():
    run_project_script(
        "scripts/finetune_asr.py",
        "--stage", "materialize",
        "--manifest-dir", MANIFEST_DIR,
        "--materialized-dir", MATERIALIZED_DIR,
    )
else:
    print("Materialized manifests already exist; reusing:", MATERIALIZED_DIR)

materialization_summary = json.loads(
    (MATERIALIZED_DIR / "materialization_summary.json").read_text(encoding="utf-8")
)
print("Materialized counts:", materialization_summary["counts"])

In [ ]:
BASE_MODEL = "vitouphy/wav2vec2-xls-r-300m-khmer"
SMOKE_DIR = RUN_ROOT / "models" / "khmer_xlsr_ddd_5000_smoke"
MODEL_DIR = RUN_ROOT / "models" / "khmer_xlsr_ddd_5000_clean"

run_project_script(
    "scripts/finetune_asr.py",
    "--stage", "validate-vocabulary",
    "--materialized-dir", MATERIALIZED_DIR,
    "--model", BASE_MODEL,
)

run_project_script(
    "scripts/finetune_asr.py",
    "--stage", "smoke",
    "--materialized-dir", MATERIALIZED_DIR,
    "--output-dir", SMOKE_DIR,
    "--model", BASE_MODEL,
)

## Full training

Run this after the smoke stage completes. On the first run, keep RESUME_TRAINING = False. To resume, attach the saved prior Kaggle output, point RESTORE_FROM in the setup cell at its asr_run folder, rerun setup and materialization cells, then set RESUME_TRAINING = True. Keep the manifest, model, output path, and checkpoint interval unchanged.

In [ ]:
RESUME_TRAINING = False
training_arguments = [
    "--stage", "full",
    "--manifest-dir", MANIFEST_DIR,
    "--materialized-dir", MATERIALIZED_DIR,
    "--output-dir", MODEL_DIR,
    "--model", BASE_MODEL,
    "--checkpoint-steps", "100",
]
if RESUME_TRAINING:
    training_arguments.append("--resume")
run_project_script("scripts/finetune_asr.py", *training_arguments)

## Evaluate on the same held-out test set

This uses the test WAVs materialized above. The report can be compared directly with the saved 1,000-row fine-tuned report and pretrained baseline, which use the same test examples.

In [ ]:
import shutil

test_manifest = MATERIALIZED_DIR / "manifests" / "test.jsonl"
audio_cache = MANIFEST_DIR / "audio_cache"
audio_cache.mkdir(parents=True, exist_ok=True)
with test_manifest.open(encoding="utf-8") as manifest_file:
    for line in manifest_file:
        row = json.loads(line)
        source_index = int(row["source_row_index"])
        cached_path = audio_cache / "source_{:06d}.wav".format(source_index)
        if not cached_path.exists():
            shutil.copy2(row["audio"], cached_path)

REPORT_DIR = RUN_ROOT / "reports" / "asr"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIVE_K_REPORT = REPORT_DIR / "finetuned_test_5000_fixed.json"
run_project_script(
    "scripts/run_asr_baseline.py",
    "--model", MODEL_DIR,
    "--manifest-dir", MANIFEST_DIR,
    "--split", "test",
    "--max-samples", "0",
    "--audio-cache-dir", audio_cache,
    "--output", FIVE_K_REPORT,
)

def show_metrics(label, path):
    report = json.loads(Path(path).read_text(encoding="utf-8"))
    metrics = report["metrics"]
    print(label, {
        key: round(100 * metrics[key], 2)
        for key in ("cer", "wer", "grapheme_cluster_cer")
    })

show_metrics("Pretrained baseline", PROJECT_ROOT / "reports" / "asr" / "baseline_test_rescored.json")
show_metrics("1,000-row fine-tune", PROJECT_ROOT / "reports" / "asr" / "finetuned_test_clean_rescored.json")
show_metrics("5,000-row fine-tune", FIVE_K_REPORT)

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

EXPORT_DIR = RUN_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_EXPORT = EXPORT_DIR / "khmer_xlsr_ddd_5000_clean_model.zip"
with ZipFile(MODEL_EXPORT, "w", compression=ZIP_DEFLATED) as archive:
    for file_path in MODEL_DIR.iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print("Model archive:", MODEL_EXPORT)
print("Evaluation report:", FIVE_K_REPORT)
print("Save a Kaggle notebook version to retain these outputs.")